# A Practical Guide to Quantitative Finance Interviews — Chapter 5
## Stochastic Process and Stochastic Calculus

Worked solutions to the stochastic-process problems in **Xinfeng Zhou's Green Book**. Each topic gets a
plain-language explanation with a small example, and every problem carries an explicit **transition graph**,
the **first-step (one-step) equations**, and a **solve-for-`n`** function with a Monte-Carlo check. Only the
Python standard library is used (`fractions` keeps every answer exact).

| § | Topic | Problems |
|---|-------|----------|
| **5.1** | Markov Chains | Gambler's ruin · Dice question · Coin triplets (A/B/C) · Color balls |

## 5.1 Markov Chains

A **Markov chain** is a random process that hops between a set of **states** $\{1,2,\dots\}$ where the next
state depends **only on the current one**, not on the full history:
$$P(X_{t+1}=j\mid X_t=i,\ X_{t-1},\dots,X_0)=P(X_{t+1}=j\mid X_t=i)=P_{ij}.$$
This "memoryless" property (the **Markov property**) is what makes the chain tractable — the present state is a
complete summary of the past.

### The transition matrix
Collect the one-step probabilities into a matrix $P$ whose entry $P_{ij}$ is the chance of moving from state
$i$ to state $j$. Every **row is a probability distribution**, so each row sums to $1$:
$$\sum_{j} P_{ij}=1\quad\text{for every }i.$$

*Simple example (weather).* States $\{S=\text{sunny},\ R=\text{rainy}\}$. Say a sunny day is followed by sun
$80\%$ of the time and a rainy day by rain $60\%$:
$$P=\begin{pmatrix}P_{SS}&P_{SR}\\P_{RS}&P_{RR}\end{pmatrix}=\begin{pmatrix}0.8&0.2\\0.4&0.6\end{pmatrix}.$$

### The probability of a path
Because each step only looks at the current state, the probability of a whole **path** is the initial
probability times the product of the one-step probabilities along it:
$$P(X_0=i_0,X_1=i_1,\dots,X_n=i_n)=P(X_0=i_0)\,P_{i_0 i_1}P_{i_1 i_2}\cdots P_{i_{n-1}i_n}.$$
*Example.* Starting sunny, the path $S\to S\to R$ has probability $P_{SS}\,P_{SR}=0.8\times0.2=0.16$.

To jump several steps at once, **multiply the matrix**: the $n$-step probabilities are the entries of $P^{n}$,
$$P(X_{t+n}=j\mid X_t=i)=\big(P^{n}\big)_{ij}.$$
For the weather chain, $P^{2}=\begin{pmatrix}0.72&0.28\\0.56&0.44\end{pmatrix}$, so two sunny days from now is
$72\%$ likely if today is sunny.

### The transition graph
The same information draws as a **directed graph**: one node per state, and an arrow $i\to j$ labeled with
$P_{ij}$ for every non-zero transition (a self-loop $i\to i$ when $P_{ii}>0$). The weather chain:

```
          0.2  (S -> R)
       ┌──────────────►┐
  0.8 ⟲│  (S)     (R)  │⟲ 0.6
       └◄──────────────┘
          0.4  (R -> S)
```

Reading a graph and reading the matrix are interchangeable; the graph makes the *structure* (which states can
reach which) jump out, which is exactly what the next idea needs.

### Classification of states
- **Accessible / communicating.** $j$ is *accessible* from $i$ if some path of positive probability leads
  $i\to\cdots\to j$. If they are accessible from each other, they **communicate**. A chain in which every state
  communicates with every other is **irreducible**.
- **Recurrent vs. transient.** From a **recurrent** state the chain is *certain* to return eventually; from a
  **transient** state there is positive probability it never comes back. In a finite chain, once it leaves a
  transient state enough times it never returns.
- **Absorbing.** A state $i$ with $P_{ii}=1$ is **absorbing** — once entered, the chain stays forever. (In the
  weather example neither state is absorbing.)
- **Periodic vs. aperiodic.** A state has **period** $d$ if returns are only possible at multiples of $d$
  steps; $d=1$ is **aperiodic**. A self-loop makes a state aperiodic.

The problems below all reduce to the most useful special case in interviews: chains with absorbing states.

### Absorbing Markov chains
A chain is **absorbing** if (1) it has at least one absorbing state and (2) every state can reach one. Order
the states with the **transient** ones first and the **absorbing** ones last; the matrix then splits into the
**canonical form**
$$P=\begin{pmatrix}Q & R\\[2pt] \mathbf 0 & I\end{pmatrix},$$
where $Q$ (transient$\to$transient) and $R$ (transient$\to$absorbing) hold everything that can still change.
The eventual fate — *which* absorbing state, and *how long* until it is reached — is what we solve for.

### Equations for absorption probability (first-step analysis)
Let $a_i=P(\text{absorbed in a chosen target set}\mid X_0=i)$. Condition on the **first step** and use the
Markov property: from $i$ you land in $j$ with probability $P_{ij}$, then face the *same* question from $j$.
$$\boxed{\,a_i=\sum_{j}P_{ij}\,a_j\,}\qquad\text{with } a_i=1 \text{ on target absorbing states, } a_i=0 \text{ on the others.}$$
In matrix form (only the transient unknowns), $a=Qa+R\mathbf 1_{\text{target}}$, i.e. $(I-Q)\,a=R\mathbf 1_{\text{target}}$.

### Equations for expected time to absorption
Let $t_i=E[\text{steps until absorption}\mid X_0=i]$. The first step *costs 1*, then you continue from wherever
you land:
$$\boxed{\,t_i=1+\sum_{j\ \text{transient}}P_{ij}\,t_j\,}\qquad t_i=0 \text{ on absorbing states.}$$
In matrix form $(I-Q)\,t=\mathbf 1$. The inverse $N=(I-Q)^{-1}$ is the **fundamental matrix**: $N_{ij}$ is the
expected number of visits to $j$ starting from $i$, so $t=N\mathbf 1$ (row sums) and the absorption
probabilities are $B=NR$. The code cell below solves both systems exactly with rational arithmetic, and every
problem in this section is an instance of these two boxed equations.

In [1]:
from fractions import Fraction

def solve_linear(A, b):
    """Solve the linear system A x = b exactly over the rationals (Gaussian elimination with Fractions)."""
    n = len(A)
    M = [[Fraction(A[i][j]) for j in range(n)] + [Fraction(b[i])] for i in range(n)]
    for col in range(n):
        piv = next(r for r in range(col, n) if M[r][col] != 0)     # nonzero pivot
        M[col], M[piv] = M[piv], M[col]
        pv = M[col][col]
        M[col] = [v / pv for v in M[col]]
        for r in range(n):
            if r != col and M[r][col] != 0:
                f = M[r][col]
                M[r] = [M[r][k] - f * M[col][k] for k in range(n + 1)]
    return [M[i][n] for i in range(n)]

def absorbing_analysis(P, states, targets):
    """P: dict {state: {next_state: prob}} with rows summing to 1. An absorbing state has P[s]=={s:1}.
    Returns (absorb_prob, expected_time), each a dict over the transient states:
      absorb_prob[i] = P(absorbed in `targets` | start i)     solves (I-Q) a = R*1_target
      expected_time[i] = E[steps to absorption | start i]     solves (I-Q) t = 1
    -- the two boxed first-step equations from the notes above."""
    absorbing = [s for s in states if P[s].get(s, 0) == 1 and len(P[s]) == 1]
    transient = [s for s in states if s not in absorbing]
    m = len(transient)
    ImQ = [[Fraction(i == j) - Fraction(P[transient[i]].get(transient[j], 0)) for j in range(m)]
           for i in range(m)]
    t = solve_linear(ImQ, [Fraction(1)] * m)
    r = [sum(Fraction(P[transient[i]].get(s, 0)) for s in targets) for i in range(m)]
    a = solve_linear(ImQ, r)
    return {transient[i]: a[i] for i in range(m)}, {transient[i]: t[i] for i in range(m)}

# quick self-test on the 2-state weather chain (no absorbing states -> just checks the linear solver)
demo = solve_linear([[Fraction(1), Fraction(-1)], [Fraction(2), Fraction(1)]], [Fraction(0), Fraction(3)])
print("linear-solver sanity check x =", demo, "(expected [1, 1])")

linear-solver sanity check x = [Fraction(1, 1), Fraction(1, 1)] (expected [1, 1])


### 5.1.1 Gambler's ruin problem

**Problem.** Player $M$ has \$1 and player $N$ has \$2; each game the winner takes \$1 from the loser. $M$ (the
better player) wins each game with probability $\tfrac23$. They play until someone is broke. What is
$P(M\text{ wins})$?

**State space.** The two stakes always sum to \$3, so a single number — $M$'s money $m\in\{0,1,2,3\}$ —
describes everything. States $0$ (M broke) and $3$ (M has it all) are **absorbing**.

**Transition graph.** A birth–death chain: from an interior state $M$ steps **up** \$1 with probability
$\tfrac23$ (he wins the game) or **down** \$1 with probability $\tfrac13$; the ends absorb ($\;⟲1$ = self-loop
of probability $1$).
```
    1 ⟲                               1 ⟲
         1/3          1/3
       ◄──────     ◄──────
  [0]*        [1]         [2]         [3]*
                 ──────►      ──────►
                   2/3          2/3
   edges:  [0]* ⟲1              [3]* ⟲1
           [1] ─1/3►[0]   [1] ─2/3►[2]
           [2] ─1/3►[1]   [2] ─2/3►[3]
```
**Transition matrix** (rows/cols $0,1,2,3$):
$$P=\begin{pmatrix}1&0&0&0\\ \tfrac13&0&\tfrac23&0\\ 0&\tfrac13&0&\tfrac23\\ 0&0&0&1\end{pmatrix}.$$

**First-step equations** for $a_m=P(\text{reach }3\mid m)$, with $a_0=0,\ a_3=1$:
$$a_1=\tfrac13 a_0+\tfrac23 a_2=\tfrac23 a_2,\qquad a_2=\tfrac13 a_1+\tfrac23 a_3=\tfrac13 a_1+\tfrac23.$$
Substituting, $a_1=\tfrac23(\tfrac13 a_1+\tfrac23)=\tfrac29 a_1+\tfrac49\Rightarrow\tfrac79 a_1=\tfrac49$, so
$$a_1=\boxed{\tfrac47}\approx0.571,\qquad a_2=\tfrac67.$$
Even starting with only one dollar, $M$'s per-game edge $p=\tfrac23>\tfrac12$ makes him the overall
favorite, since $a_1=\tfrac47>\tfrac12$. More generally, for win probability $p\neq\tfrac12$ with ratio
$r=\tfrac{1-p}{p}$, starting stake $i$, and target $N$, the classic gambler's-ruin formula is
$$a_i=\frac{1-r^{\,i}}{1-r^{\,N}},$$
which here ($r=\tfrac12,\ i=1,\ N=3$) gives $a_1=\dfrac{1-(1/2)^{1}}{1-(1/2)^{3}}=\dfrac{1/2}{7/8}=\tfrac47$.

In [2]:
from fractions import Fraction
import random

def gamblers_ruin_chain(start, total, p_win):
    """Build the gambler's-ruin Markov chain on money in {0,...,total} (0 and `total` absorbing) and return
    P(reach `total` before 0 | start) via the absorption equations."""
    p = Fraction(p_win)
    P = {0: {0: Fraction(1)}, total: {total: Fraction(1)}}
    for m in range(1, total):
        P[m] = {m - 1: 1 - p, m + 1: p}
    a, _ = absorbing_analysis(P, list(range(total + 1)), targets={total})
    return a[start]

def gamblers_ruin_closed(i, N, p):
    """Closed form: fair game -> i/N, else (1 - r^i)/(1 - r^N) with r = (1-p)/p."""
    p = Fraction(p)
    if p == Fraction(1, 2):
        return Fraction(i, N)
    r = (1 - p) / p
    return (1 - r ** i) / (1 - r ** N)

def gamblers_ruin_sim(start=1, total=3, p_win=2 / 3, trials=200000, seed=0):
    rng = random.Random(seed); wins = 0
    for _ in range(trials):
        m = start
        while 0 < m < total:
            m += 1 if rng.random() < p_win else -1
        wins += (m == total)
    return wins / trials

ans = gamblers_ruin_chain(1, 3, Fraction(2, 3))
print("P(M wins | $1, target $3, p=2/3) =", ans, "=", float(ans))
print("  closed form:", gamblers_ruin_closed(1, 3, Fraction(2, 3)), " | simulated:", round(gamblers_ruin_sim(), 4))
for st in (1, 2):
    print(f"  start ${st}: chain {gamblers_ruin_chain(st, 3, Fraction(2,3))}")

P(M wins | $1, target $3, p=2/3) = 4/7 = 0.5714285714285714
  closed form: 4/7  | simulated: 0.5703
  start $1: chain 4/7
  start $2: chain 6/7


### 5.1.2 Dice question

**Problem.** Two dice are rolled repeatedly and the sums recorded. Player $A$ bets a sum of **$12$** appears
first; player $B$ bets **two consecutive $7$s** appear first. What is $P(A\text{ wins})$?

**Per-roll probabilities.** For two dice, $P(\text{sum}=12)=\tfrac1{36}$, $P(\text{sum}=7)=\tfrac6{36}$, and
"anything else" $=\tfrac{29}{36}$.

**States.** $B$ needs *two $7$s in a row*, so the chain must remember whether the **previous roll was a $7$**:
- $S_0$ — last roll was not a $7$ (also the start);
- $S_1$ — last roll **was** a $7$ (one down, one to go);
- $A$ — a $12$ has appeared ($A$ wins), absorbing;
- $B$ — two $7$s in a row ($B$ wins), absorbing.

**Transition graph.**
```
             29/36 (other)
            ┌──────────┐
            ▼          │
   [A]* ◄─1/36─ [S0] ─6/36─► [S1] ─6/36─► [B]*
                  ▲            │  │
                  └────29/36───┘  └─1/36─► [A]*
   edges:  [S0] ─1/36►[A]   [S0] ─6/36►[S1]   [S0] ─29/36►[S0]
           [S1] ─1/36►[A]   [S1] ─6/36►[B]    [S1] ─29/36►[S0]
```
**First-step equations** for $a=P(A\text{ wins})$ from each state ($a_A=1,\ a_B=0$):
$$a_{S_0}=\tfrac1{36}+\tfrac6{36}a_{S_1}+\tfrac{29}{36}a_{S_0},\qquad
a_{S_1}=\tfrac1{36}+\tfrac6{36}\cdot0+\tfrac{29}{36}a_{S_0}.$$
From the first, $7\,a_{S_0}=1+6\,a_{S_1}$; from the second, $36\,a_{S_1}=1+29\,a_{S_0}$. Eliminating
$a_{S_1}$ gives $78\,a_{S_0}=42$, so
$$P(A\text{ wins})=a_{S_0}=\boxed{\tfrac{7}{13}}\approx0.538.$$
Surprisingly $A$ is the favorite: although a $12$ ($\tfrac1{36}$) is far rarer than a $7$ ($\tfrac6{36}$), $B$
must hit the $7$ **twice in a row**, and the frequent "other" rolls keep resetting that streak back to $S_0$.

In [3]:
from fractions import Fraction
import random

def dice_race_prob():
    """P(A wins): a sum of 12 appears before two consecutive 7s. State S0 = last roll not a 7, S1 = last a 7."""
    p12, p7 = Fraction(1, 36), Fraction(6, 36)
    po = 1 - p12 - p7
    P = {'S0': {'A': p12, 'S1': p7, 'S0': po},
         'S1': {'A': p12, 'B': p7, 'S0': po},
         'A': {'A': Fraction(1)}, 'B': {'B': Fraction(1)}}
    a, _ = absorbing_analysis(P, ['S0', 'S1', 'A', 'B'], targets={'A'})
    return a['S0']

def dice_race_sim(trials=300000, seed=0):
    rng = random.Random(seed); Awin = 0
    for _ in range(trials):
        prev7 = False
        while True:
            s = rng.randint(1, 6) + rng.randint(1, 6)
            if s == 12:
                Awin += 1; break
            if s == 7:
                if prev7:
                    break            # two 7s in a row -> B wins
                prev7 = True
            else:
                prev7 = False
    return Awin / trials

ans = dice_race_prob()
print("P(A wins: 12 before two consecutive 7s) =", ans, "=", float(ans))
print("  simulated:", round(dice_race_sim(), 4))

P(A wins: 12 before two consecutive 7s) = 7/13 = 0.5384615384615384
  simulated: 0.5376


### 5.1.3 Coin triplets

Three linked questions about patterns in fair-coin tosses. The unifying trick is a Markov chain whose state is
**how much of the target pattern the recent tosses have already built** — the longest suffix of what we have
tossed that is a prefix of the pattern. From each state a toss either extends the progress, completes the
pattern, or falls back (possibly using an **overlap**).

#### Part A — expected tosses to see a pattern

**Question.** How many tosses on average to first see **HHH**? And to first see **THH**?

**Pattern HHH.** States by progress $\varnothing,\text{H},\text{HH}$ (then HHH ends it). A tail **T** wipes all
progress (T is not a prefix of HHH), so it always resets to $\varnothing$:
```
   [∅] ─H(1/2)─► [H] ─H(1/2)─► [HH] ─H(1/2)─► (HHH) done
    ▲             │             │
    └────T(1/2)───┴────T(1/2)───┘   (any T resets to ∅)
   (∅ also self-loops on T)
```
With $e_s=E[\text{tosses to HHH}\mid s]$:
$$e_{HH}=1+\tfrac12\cdot0+\tfrac12 e_\varnothing,\quad e_{H}=1+\tfrac12 e_{HH}+\tfrac12 e_\varnothing,\quad
e_\varnothing=1+\tfrac12 e_H+\tfrac12 e_\varnothing.$$
Solving gives $e_\varnothing=\boxed{14}$ tosses $(=2+4+8)$.

**Pattern THH.** Now a **T never fully wastes progress**: after reaching TH, a tail returns you to state T (that
T can start a fresh THH), and once in T another T just stays in T:
```
   [∅] ─T(1/2)─► [T] ─H(1/2)─► [TH] ─H(1/2)─► (THH) done
    ▲ ⟲H(1/2)    ▲ ⟲T(1/2)      │
    │            └──────T(1/2)───┘   (TH --T--> T, not ∅)
    └────────────  (∅ --H--> ∅)
```
$$e_{TH}=1+\tfrac12\cdot0+\tfrac12 e_{T},\quad e_{T}=1+\tfrac12 e_{TH}+\tfrac12 e_{T},\quad
e_\varnothing=1+\tfrac12 e_T+\tfrac12 e_\varnothing.$$
Solving gives $e_{T}=6,\ e_{TH}=4$, and $e_\varnothing=\boxed{8}$ tosses.

**Why HHH is slower ($14$ vs $8$).** HHH **overlaps itself**: a run of heads keeps every head useful, but a
single tail throws away *all* the built-up heads at once. THH cannot self-overlap on its head part, so a tail
only ever costs you the last step. (Conway's shortcut: the expected wait is $\sum_k 2^{k}$ over the lengths $k$
where the pattern's prefix equals its suffix — HHH matches at $k=1,2,3\Rightarrow2{+}4{+}8{=}14$; THH only at
$k=3\Rightarrow8$.)

In [4]:
from fractions import Fraction
import random

def expected_wait(pattern, p_head=Fraction(1, 2)):
    """Expected number of tosses to first see `pattern` (a string of 'H'/'T'). State = longest suffix of the
    tosses so far that is a prefix of `pattern`; solved with the expected-time absorption equations."""
    L = len(pattern); pr = {'H': Fraction(p_head), 'T': 1 - Fraction(p_head)}
    prefixes = [pattern[:k] for k in range(L)]                 # '', p0, p0p1, ...
    def nxt(s, c):
        t = s + c
        if t == pattern:
            return pattern
        for k in range(min(len(t), L - 1), -1, -1):           # longest suffix that is a prefix
            if t[len(t) - k:] == pattern[:k]:
                return pattern[:k]
    states = prefixes + [pattern]
    P = {s: {} for s in prefixes}
    for s in prefixes:
        for c in 'HT':
            d = nxt(s, c); P[s][d] = P[s].get(d, 0) + pr[c]
    P[pattern] = {pattern: Fraction(1)}
    _, t = absorbing_analysis(P, states, targets={pattern})
    return t['']

def wait_sim(pattern, trials=200000, seed=0):
    rng = random.Random(seed); tot = 0
    for _ in range(trials):
        seq = ''; n = 0
        while not seq.endswith(pattern):
            seq += 'H' if rng.random() < 0.5 else 'T'; n += 1
        tot += n
    return tot / trials

for pat in ('HHH', 'THH'):
    print(f"E[tosses to {pat}] = {expected_wait(pat)}   (simulated {wait_sim(pat):.3f})")
print("also, e.g. E[HT] =", expected_wait('HT'), ", E[HTH] =", expected_wait('HTH'))

E[tosses to HHH] = 14   (simulated 14.000)
E[tosses to THH] = 8   (simulated 7.983)
also, e.g. E[HT] = 4 , E[HTH] = 10


#### Part B — which comes first, HHH or THH?

**Question.** Keep flipping until **either** HHH or THH appears. What is $P(\text{HHH first})$?

**Combined chain.** Track the progress that is relevant to *both* patterns; the reachable states are
$\varnothing,\text{H},\text{HH},\text{T},\text{TH}$, with absorbing wins **HHH** and **THH**.
```
   [∅] ─H─► [H] ─H─► [HH] ─H─► (HHH win)
    │        │         │
    │T       │T        │T
    ▼        ▼         ▼
   [T] ◄─────┴─────────┘        (any T from the H-side drops to T)
    │⟲T
    │H
    ▼
   [TH] ─H─► (THH win)   ,   [TH] ─T─► [T]
   (every edge has probability 1/2)
```
**First-step equations** for $p_s=P(\text{HHH first}\mid s)$, with $p_{HHH}=1,\ p_{THH}=0$:
$$p_{HH}=\tfrac12\cdot1+\tfrac12 p_T,\quad p_H=\tfrac12 p_{HH}+\tfrac12 p_T,\quad p_\varnothing=\tfrac12 p_H+\tfrac12 p_T,$$
$$p_{TH}=\tfrac12\cdot0+\tfrac12 p_T,\quad p_T=\tfrac12 p_{TH}+\tfrac12 p_T.$$
The last pair forces $p_T=p_{TH}=0$: **once any tail appears, THH is unstoppable before HHH.** Then
$p_{HH}=\tfrac12,\ p_H=\tfrac14$, and
$$P(\text{HHH first})=p_\varnothing=\tfrac12\cdot\tfrac14+\tfrac12\cdot0=\boxed{\tfrac18}.$$
The intuition: to get HHH first you essentially must throw **HHH on the very first three tosses** ($\tfrac18$).
Any earlier tail plants the "T" that THH needs, and THH then completes on the next two heads before a third
head could ever extend to HHH. So THH wins with probability $\tfrac78$.

In [5]:
from fractions import Fraction
import random

def prob_pattern_before(A, B, p_head=Fraction(1, 2)):
    """P(pattern A appears strictly before pattern B) in a fair(ish)-coin stream, via first-step analysis on
    the combined progress states (longest suffix that is a prefix of A or of B)."""
    pr = {'H': Fraction(p_head), 'T': 1 - Fraction(p_head)}
    prefixes = {''}
    for Q in (A, B):
        for k in range(1, len(Q)):
            prefixes.add(Q[:k])
    def nxt(s, c):
        t = s + c
        if t.endswith(A): return 'AW'
        if t.endswith(B): return 'BW'
        for k in range(len(t), -1, -1):
            if t[len(t) - k:] in prefixes:
                return t[len(t) - k:]
    states = list(prefixes) + ['AW', 'BW']
    P = {s: {} for s in prefixes}
    for s in prefixes:
        for c in 'HT':
            d = nxt(s, c); P[s][d] = P[s].get(d, 0) + pr[c]
    P['AW'] = {'AW': Fraction(1)}; P['BW'] = {'BW': Fraction(1)}
    a, _ = absorbing_analysis(P, states, targets={'AW'})
    return a['']

def race_sim(A, B, trials=200000, seed=0):
    rng = random.Random(seed); Aw = 0
    for _ in range(trials):
        seq = ''
        while True:
            seq += 'H' if rng.random() < 0.5 else 'T'
            if seq.endswith(A): Aw += 1; break
            if seq.endswith(B): break
    return Aw / trials

p = prob_pattern_before('HHH', 'THH')
print("P(HHH before THH) =", p, "=", float(p), "  | P(THH first) =", 1 - p)
print("  simulated P(HHH first):", round(race_sim('HHH', 'THH'), 4))

P(HHH before THH) = 1/8 = 0.125   | P(THH first) = 7/8
  simulated P(HHH first): 0.1251


#### Part C — both players choose (Penney's game)

**Question.** Player 1 picks a triplet and **announces** it; player 2 then picks a **different** triplet; whoever's
triplet appears first wins. Both are perfectly rational. What is **player 2's** probability of winning?

**Key fact — the game is non-transitive.** "Appears first" is not a total order: for any triplet player 1 names,
player 2 can name one that beats it. The optimal reply to player 1's $b_1b_2b_3$ is
$$\text{player 2}=(\overline{b_2})\,b_1 b_2\qquad(\overline{b_2}=\text{the opposite of the middle symbol}),$$
which "hooks" onto the front of player 1's pattern. Because player 2 moves second, they always hold the whip
hand — the only question is how badly, and player 1 (rational) picks the triplet that **minimizes** player 2's
edge.

**The full odds table** (player 2's win probability with the optimal reply — the code computes every entry with
`prob_pattern_before`):

| player 1 | player 2 (best reply) | P(player 2 wins) |
|---|---|---|
| HHH | THH | $7/8$ |
| HHT | THH | $3/4$ |
| HTH | HHT | $2/3$ |
| HTT | HHT | $2/3$ |
| THH | TTH | $2/3$ |
| THT | TTH | $2/3$ |
| TTH | HTT | $3/4$ |
| TTT | HTT | $7/8$ |

**Minimax.** A rational player 1 avoids HHH/HHT/TTH/TTT (which hand player 2 $\tfrac34$ or $\tfrac78$) and picks
one of HTH, HTT, THH, THT — the best defense — holding player 2 down to the smallest available edge:
$$P(\text{player 2 wins})=\boxed{\tfrac23}.$$
So even against a perfect opponent, going second is worth $2:1$ odds. (This is why you should always let your
friend call their triplet first.)

In [6]:
from fractions import Fraction
import itertools

pats = [''.join(t) for t in itertools.product('HT', repeat=3)]

def best_reply(p1):
    """Player 2's payoff-maximizing triplet against announced p1, and the resulting P(player 2 wins)."""
    return max(((b, prob_pattern_before(b, p1)) for b in pats if b != p1), key=lambda kv: kv[1])

print("Penney's game -- player 1 announces, player 2 replies optimally:")
rows = []
for p1 in pats:
    b, pr = best_reply(p1)
    rows.append((p1, b, pr))
    print(f"   P1={p1}  ->  P2={b}   P2 wins {pr} = {float(pr):.3f}")

p1_opt, p2_pick, p2_prob = min(rows, key=lambda r: r[2])     # rational P1 minimizes P2's edge
print(f"\nrational P1 plays {p1_opt} (a minimax choice); P2 replies {p2_pick} and wins "
      f"{p2_prob} = {float(p2_prob):.4f}")

Penney's game -- player 1 announces, player 2 replies optimally:
   P1=HHH  ->  P2=THH   P2 wins 7/8 = 0.875
   P1=HHT  ->  P2=THH   P2 wins 3/4 = 0.750
   P1=HTH  ->  P2=HHT   P2 wins 2/3 = 0.667
   P1=HTT  ->  P2=HHT   P2 wins 2/3 = 0.667
   P1=THH  ->  P2=TTH   P2 wins 2/3 = 0.667
   P1=THT  ->  P2=TTH   P2 wins 2/3 = 0.667
   P1=TTH  ->  P2=HTT   P2 wins 3/4 = 0.750
   P1=TTT  ->  P2=HTT   P2 wins 7/8 = 0.875

rational P1 plays HTH (a minimax choice); P2 replies HHT and wins 2/3 = 0.6667


### 5.1.4 Color balls

**Problem.** A box holds $n$ balls, each initially a **different** color. Repeatedly: pick an ordered pair of
balls at random, **repaint the first to match the second**, and return both. What is the expected number of
steps until **all balls share one color**?

**State = the partition of colors.** By symmetry only the multiset of color-group **sizes** matters (a partition
of $n$). Start at $(1,1,\dots,1)$; absorb at $(n)$. Take $n=3$ (groups written largest-first):

- From $(1,1,1)$: whichever ordered pair is chosen, one ball adopts another's color, giving two-of-a-kind — so
  it moves to $(2,1)$ **with probability 1**.
- From $(2,1)$ (colors $X,X,Y$): of the $3\times2=6$ ordered pairs, the two that repaint the lone $Y$ ball to
  $X$ — i.e. (first $=Y$, second $=$ an $X$) — finish the job $\to(3)$; the other four leave a $2\!-\!1$ split.
  So $(2,1)\to(3)$ w.p. $\tfrac26=\tfrac13$ and stays at $(2,1)$ w.p. $\tfrac23$.

**Transition graph ($n=3$).**
```
                       2/3 (self)
                      ┌────────┐
                      ▼        │
   [(1,1,1)] ──1──► [(2,1)] ───┘ ──1/3──► [(3)]*
```
**Expected-time equations** ($t_{(3)}=0$):
$$t_{(2,1)}=1+\tfrac23 t_{(2,1)}\ \Rightarrow\ t_{(2,1)}=3,\qquad
t_{(1,1,1)}=1+t_{(2,1)}=\boxed{4}.$$

**General $n$.** Running the same partition chain for each $n$ (the code builds it and solves
$(I-Q)t=\mathbf 1$ exactly) gives $1,4,9,16,25,\dots$ — that is,
$$E[\text{steps}]=\boxed{(n-1)^{2}}.$$
Intuitively the process is a *voter model* on the complete graph: the count of monochromatic pairs
$\Phi=\sum_c\binom{X_c}{2}$ drifts up by $E[\Delta\Phi]=1-\tfrac{2\Phi}{n(n-1)}$ each step, mean-reverting toward
the absorbed value $\binom n2$; solving the chain turns that drift into the clean $(n-1)^2$.

In [7]:
from fractions import Fraction
import random

def _partitions(n, mx=None):
    """All integer partitions of n as tuples sorted largest-first."""
    if mx is None: mx = n
    if n == 0:
        yield (); return
    for first in range(min(n, mx), 0, -1):
        for rest in _partitions(n - first, first):
            yield (first,) + rest

def color_balls_expected_exact(n):
    """Exact expected steps until one color remains, by building the partition-size Markov chain (pick an
    ordered pair of the n(n-1) ordered pairs; move one ball from its group to the other's) and solving the
    expected-time equations."""
    states = list(_partitions(n)); absorb = (n,); tot = n * (n - 1)
    P = {p: {} for p in states}
    for p in states:
        if p == absorb:
            P[p] = {p: Fraction(1)}; continue
        acc, k = {}, len(p)
        for a in range(k):
            for b in range(k):
                if a == b:
                    w = p[a] * (p[a] - 1)                       # both balls same group -> no change
                    if w: acc[p] = acc.get(p, 0) + w
                else:
                    w = p[a] * p[b]                             # first from group a, second from group b
                    if not w: continue
                    ns = list(p); ns[a] -= 1; ns[b] += 1
                    ns = tuple(sorted((x for x in ns if x > 0), reverse=True))
                    acc[ns] = acc.get(ns, 0) + w
        P[p] = {s: Fraction(v, tot) for s, v in acc.items()}
    _, t = absorbing_analysis(P, states, targets={absorb})
    return t[tuple([1] * n)]

def color_balls_closed(n):
    """Closed form."""
    return (n - 1) ** 2

def color_balls_sim(n, trials=20000, seed=0):
    rng = random.Random(seed); tot = 0
    for _ in range(trials):
        colors = list(range(n)); steps = 0
        while len(set(colors)) > 1:
            i = rng.randrange(n)
            j = rng.randrange(n)
            while j == i:
                j = rng.randrange(n)
            colors[i] = colors[j]; steps += 1        # first ball repainted to second's color
        tot += steps
    return tot / trials

print("n : exact (partition chain) | (n-1)^2 | simulated")
for n in range(2, 7):
    exact = color_balls_expected_exact(n)
    sim = f"{color_balls_sim(n):.2f}" if n <= 5 else "  -"
    print(f"{n} : {str(exact):>4}                      | {color_balls_closed(n):>3}     | {sim}")

n : exact (partition chain) | (n-1)^2 | simulated
2 :    1                      |   1     | 1.00
3 :    4                      |   4     | 3.98
4 :    9                      |   9     | 8.94
5 :   16                      |  16     | 16.00
6 :   25                      |  25     |   -


---
*More of Chapter 5 as I keep reading.*